# RiskPulse: embeddings + CPU XGBoost training

This notebook precomputes frozen `all-MiniLM-L6-v2` embeddings on Kaggle's GPU when available, then fits a small XGBoost regressor on CPU. The target is the absolute next-trading-day stock price move derived from public event/news market labels. It is a proxy for event severity, not a portfolio loss forecast.

Attach the EDT dataset (the `Trading_benchmark/evaluate_news.json` file) as a private Kaggle input. The notebook scans `/kaggle/input` for that filename. If it is missing, set `DATA_FILE` to the mounted file shown by the discovery cell. Do not paste API keys into this notebook.

Expected outputs under `/kaggle/working/models`: `impact_xgb.json`, `impact_calibration.json`, `embedding_cache.npz`, and `training_report.json`. Download these from the notebook Output panel. Copy the first two to this repo's `models/` directory to enable model inference.

The source dataset card reports about 303,893 market-labeled news records and warns that automatically assigned tickers contain errors. This notebook filters missing tickers/prices and requires the ticker symbol to appear in the text or a market identifier to appear. Review the filter counts and validation metrics before trusting the model.

In [ ]:
%pip -q install sentence-transformers "xgboost>=2" "scikit-learn>=1.3"
import json, math, os, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

files = list(Path('/kaggle/input').rglob('*'))
matches = [p for p in files if p.name.lower() == 'evaluate_news.json']
print('EDT candidate files:', matches[:5])
if matches:
    DATA_FILE = matches[0]
else:
    import requests
    DATA_FILE = Path('/kaggle/working/data/evaluate_news.json')
    DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
    url = 'https://huggingface.co/datasets/agungpambudi/corporate-event-detection/resolve/main/datasets/benchmark/evaluate_news.json?download=true'
    expected = int(requests.head(url, allow_redirects=True, timeout=60).headers.get('content-length', 0))
    if not DATA_FILE.exists() or (expected and DATA_FILE.stat().st_size != expected):
        temp = DATA_FILE.with_suffix('.json.part')
        response = requests.get(url, stream=True, timeout=(30, 180))
        response.raise_for_status()
        downloaded = 0
        with temp.open('wb') as f:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if downloaded % (128 * 1024 * 1024) < 8 * 1024 * 1024:
                        print(f'downloaded {downloaded / 1024**3:.2f} GiB', flush=True)
        if expected and downloaded != expected:
            raise IOError(f'Incomplete EDT download: {downloaded} of {expected} bytes')
        temp.replace(DATA_FILE)
print('Using', DATA_FILE, f'({DATA_FILE.stat().st_size / 1024**2:.1f} MiB)')
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# Normalize EDT labels. The input remains read-only under /kaggle/input.
with open(DATA_FILE, encoding='utf-8') as f:
    raw = json.load(f)
if isinstance(raw, dict):
    raw = raw.get('data', raw.get('articles', []))
print('Raw records:', len(raw))
rows = []
for item in raw:
    lab = item.get('labels') or {}
    if isinstance(lab, str):
        try: lab = json.loads(lab)
        except json.JSONDecodeError: lab = {}
    if not isinstance(lab, dict): continue
    title = str(item.get('title') or '').strip()
    body = str(item.get('text') or '').strip()[:3000]
    ticker = str(lab.get('ticker') or '').strip()
    start = lab.get('start_price_close') or lab.get('start_price_open')
    end = lab.get('end_price_1day')
    text = (title + ' ' + body).lower()
    ticker_match = bool(ticker and re.search(r'(?<![a-z0-9])' + re.escape(ticker.lower()) + r'(?![a-z0-9])', text))
    if not (ticker_match or 'nasdaq' in text or 'nyse' in text): continue
    try:
        start, end = float(start), float(end)
        move = abs((end / start - 1.0) * 100.0)
    except (TypeError, ValueError, ZeroDivisionError): continue
    if not title or not np.isfinite(move) or move > 100: continue
    rows.append({'published_at': item.get('pub_time', ''), 'ticker': ticker, 'title': title, 'summary': body, 'target_abs_return_pct': move})
data = pd.DataFrame(rows)
data['published_at'] = pd.to_datetime(data['published_at'], errors='coerce', utc=True)
data = data.dropna(subset=['published_at']).drop_duplicates(['published_at','ticker','title']).sort_values('published_at').reset_index(drop=True)
print('Usable rows after price/ticker checks:', len(data))
display(data.head(3))
if len(data) < 1000: raise ValueError('Fewer than 1,000 usable rows; verify EDT JSON structure and label fields.')

# Start with 50k to cap cache size. Set None to use all valid rows if desired.
MAX_ROWS = 50_000
if MAX_ROWS and len(data) > MAX_ROWS: data = data.tail(MAX_ROWS).reset_index(drop=True)
print('Training sample:', len(data), 'rows; cache is about', round(len(data) * 384 * 2 / 1024**2), 'MiB float16')

In [ ]:
# Checkpoint embeddings in small batches so an interrupted cell can resume.
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = SentenceTransformer(MODEL_NAME, device=device)
texts = (data['title'] + '. ' + data['summary']).tolist()
OUT = Path('/kaggle/working/models')
OUT.mkdir(parents=True, exist_ok=True)
cache_path = Path('/kaggle/working/embedding_cache_float32.dat')
state_path = Path('/kaggle/working/embedding_progress.json')
signature = {'rows': len(texts), 'model': MODEL_NAME, 'first': texts[0][:200], 'last': texts[-1][:200]}
if state_path.exists() and json.loads(state_path.read_text()).get('signature') == signature and cache_path.exists() and cache_path.stat().st_size == len(texts)*384*4:
    state = json.loads(state_path.read_text())
    start_at = min(int(state.get('completed_rows', 0)), len(texts))
    mmap_mode = 'r+'
else:
    start_at = 0
    mmap_mode = 'w+'
X_map = np.memmap(cache_path, dtype='float32', mode=mmap_mode, shape=(len(texts), 384))
if start_at < len(texts):
    started = time.time()
    for batch_start in range(start_at, len(texts), 2048):
        batch_stop = min(batch_start + 2048, len(texts))
        X_map[batch_start:batch_stop] = encoder.encode(texts[batch_start:batch_stop], batch_size=128, normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True)
        X_map.flush()
        state_tmp = state_path.with_suffix('.tmp')
        state_tmp.write_text(json.dumps({'signature': signature, 'completed_rows': batch_stop}, indent=2))
        state_tmp.replace(state_path)
        print(f'checkpointed {batch_stop:,}/{len(texts):,}')
    print('Embedding device:', device, 'minutes:', round((time.time()-started)/60, 2))
X = X_map
print('Embedding matrix:', X.shape, 'float32 cache MiB:', round(cache_path.stat().st_size/1024**2, 1))

# Hold out the latest 20% by publication time. XGBoost trains on CPU.
y_raw = data['target_abs_return_pct'].to_numpy(dtype=np.float32)
y = np.log1p(y_raw)
split = int(len(data) * 0.8)
if split < 500 or len(data) - split < 100: raise ValueError('Need >=500 train and >=100 time-ordered validation rows.')
model = XGBRegressor(n_estimators=350, max_depth=5, learning_rate=0.04, subsample=0.85, colsample_bytree=0.8, reg_lambda=5.0, objective='reg:squarederror', tree_method='hist', device='cpu', n_jobs=4, random_state=20261003)
model.fit(X[:split], y[:split], eval_set=[(X[split:], y[split:])], verbose=False)
pred = np.expm1(np.maximum(0, model.predict(X[split:])))
mae = float(mean_absolute_error(y_raw[split:], pred))
rmse = float(np.sqrt(mean_squared_error(y_raw[split:], pred)))
baseline = float(np.median(y_raw[:split]))
baseline_mae = float(mean_absolute_error(y_raw[split:], np.full_like(y_raw[split:], baseline)))
print({'time_split': str(data['published_at'].iloc[split]), 'validation_MAE_pct_points': mae, 'validation_RMSE_pct_points': rmse, 'median_baseline_MAE_pct_points': baseline_mae})

In [ ]:
# Save resumable, downloadable artifacts to Kaggle's working/output folder.
model.save_model(OUT / 'impact_xgb.json')
calibration = {'embedding_model': MODEL_NAME, 'target': 'absolute next-trading-day price move in percent', 'target_transform': 'log1p', 'p10_abs_return_pct': float(np.quantile(y_raw[:split], .10)), 'p90_abs_return_pct': float(np.quantile(y_raw[:split], .90)), 'rows': len(data), 'train_rows': split, 'validation_rows': len(data)-split, 'validation_mae_pct_points': mae, 'validation_rmse_pct_points': rmse, 'median_baseline_mae_pct_points': baseline_mae, 'validation_period_start': str(data['published_at'].iloc[split])}
(OUT / 'impact_calibration.json').write_text(json.dumps(calibration, indent=2))
np.savez_compressed(OUT / 'embedding_cache.npz', embeddings=X.astype(np.float16), target_abs_return_pct=y_raw, published_at=data['published_at'].astype(str).to_numpy())
(OUT / 'training_report.json').write_text(json.dumps(calibration, indent=2))
for p in sorted(OUT.iterdir()): print(p.name, round(p.stat().st_size / 1024**2, 2), 'MiB')
from IPython.display import FileLink, display
display(FileLink(str(OUT / 'impact_xgb.json')), FileLink(str(OUT / 'impact_calibration.json')), FileLink(str(OUT / 'embedding_cache.npz')), FileLink(str(OUT / 'training_report.json')))
cache_path.unlink(missing_ok=True)
state_path.unlink(missing_ok=True)
